# starling vs darfix -- characterisation report

This notebook loads the output of `scripts/compare_darfix.py` (its `--output-dir`) and renders, for every comparable per-pixel map, a **side-by-side + difference** panel with the difference statistics.

**This is a characterisation, not a parity test.** starling and darfix are *expected* to differ systematically because they make different modelling choices:

| stage | starling | darfix |
|---|---|---|
| background | `estimate_background` over the *n* lowest frames (mean/median/percentile) | per-frame background subtraction (`BS`, method mean/median, `background_type='Data'`) |
| peak model | per-axis **Gaussian + constant background**, Gauss-Newton | **multivariate Gaussian** rocking-curve fit with min-background |
| hot pixels | one-sided sigma-clip (`min_sigma=1.0`) | median-kernel replacement (`kernel_size`) |
| moments | intensity-weighted, **no smoothing** | intensity-weighted, **median-smoothed** (`smooth=True`) |

So the goal is to *quantify* the differences (median / 95th-pct absolute difference, Pearson r, fraction of COM pixels within one motor grid step), not to drive them to zero.

Two comparison families are produced:
* **fit-vs-fit** (`com_fit_*`, `fwhm_fit_*`) -- starling Gaussian fit vs darfix `RockingCurves`, on `starling fit_status==1 AND darfix 'Fit successful'`.
* **moment-vs-moment** (`com_mom_*`) -- both packages' intensity-weighted moments, on the starling grain mask.
* `spread_fit` -- a common geometric-mean-of-per-axis-FWHM proxy (neither package exposes a shared scalar mosaicity).

In [ ]:
import json
import os

import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# point this at the script's --output-dir
OUTPUT_DIR = os.environ.get('COMPARE_DARFIX_OUTDIR', '/tmp/cmp_out')

with open(os.path.join(OUTPUT_DIR, 'report.json')) as f:
    report = json.load(f)

print('scan          :', report.get('scan'))
print('motors        :', report.get('motor_names'))
print('motor steps   :', report.get('motor_steps'))
print('n compared px :', report.get('n_compared_px'))
print('starling      :', report.get('starling'))
print('darfix        :', report.get('darfix'))

## Statistics table

One row per compared map. `frac_within_1_step` is the fraction of pixels whose absolute difference is within one motor grid step (COM only; `None` for widths).

In [ ]:
maps = report['maps']
cols = ['n_px', 'median_abs_diff', 'p95_abs_diff', 'mean_diff', 'std_diff',
        'pearson_r', 'frac_within_1_step']

def fmt(x):
    if x is None:
        return '     -'
    if isinstance(x, float):
        return f'{x:.4g}'
    return str(x)

hdr = f"{'map':<16} " + ' '.join(f'{c:>16}' for c in cols)
print(hdr)
print('-' * len(hdr))
for name, st in maps.items():
    if not isinstance(st, dict) or 'n_px' not in st:
        print(f'{name:<16} {st}')
        continue
    print(f'{name:<16} ' + ' '.join(f'{fmt(st.get(c)):>16}' for c in cols))

## Per-map panels: starling | darfix | difference

Off-mask (non-compared) pixels are drawn grey. The difference panel uses a symmetric diverging colormap centred on zero, so a systematic bias shows as an overall colour tint and random scatter shows as speckle.

In [ ]:
def load_maps(h5path):
    out = {'starling': {}, 'darfix': {}, 'difference': {}}
    with h5py.File(h5path, 'r') as f:
        for grp in ('starling', 'darfix'):
            if grp not in f:
                continue
            for fam in f[grp]:
                node = f[grp][fam]
                if isinstance(node, h5py.Group):
                    for nm in node:
                        out[grp][f'{fam}_{nm}'] = node[nm][()]
                else:
                    out[grp][fam] = node[()]
        if 'difference' in f:
            for k in f['difference']:
                out['difference'][k] = f['difference'][k][()]
    return out

M = load_maps(os.path.join(OUTPUT_DIR, 'maps.h5'))
print('difference maps available:', sorted(M['difference'].keys()))

In [ ]:
def panel(key):
    if key not in M['starling'] or key not in M['darfix'] or key not in M['difference']:
        print(f'{key}: not available in maps.h5')
        return
    a = np.asarray(M['starling'][key], float)
    b = np.asarray(M['darfix'][key], float)
    d = np.asarray(M['difference'][key], float)
    finite = np.isfinite(a) & np.isfinite(b)
    if not finite.any():
        print(f'{key}: no finite pixels')
        return
    vlo = float(np.nanmin([np.nanmin(a), np.nanmin(b)]))
    vhi = float(np.nanmax([np.nanmax(a), np.nanmax(b)]))
    dmax = float(np.nanmax(np.abs(d))) if np.isfinite(d).any() else 1.0
    dmax = dmax if dmax > 0 else 1.0
    st = maps.get(key, {})

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    for ax, arr, title, cmap, norm in (
        (axes[0], a, 'starling', 'viridis', None),
        (axes[1], b, 'darfix', 'viridis', None),
        (axes[2], d, 'starling - darfix', 'RdBu_r',
         TwoSlopeNorm(vcenter=0.0, vmin=-dmax, vmax=dmax)),
    ):
        cmap_obj = plt.get_cmap(cmap).copy()
        cmap_obj.set_bad('0.6')
        kw = dict(cmap=cmap_obj, interpolation='nearest', origin='lower')
        if norm is not None:
            kw['norm'] = norm
        else:
            kw['vmin'], kw['vmax'] = vlo, vhi
        im = ax.imshow(arr, **kw)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.8)
    fig.suptitle(
        f"{key}   n={st.get('n_px')}  med|d|={fmt(st.get('median_abs_diff'))}  "
        f"p95|d|={fmt(st.get('p95_abs_diff'))}  r={fmt(st.get('pearson_r'))}",
        fontsize=11)
    plt.show()

for key in maps:
    panel(key)

## Correlation scatter (fit-vs-fit COM)

A tight diagonal means the two packages place the peak centre at the same motor position pixel-by-pixel; an offset diagonal is a bias (e.g. from the different background estimate); scatter width reflects per-pixel disagreement.

In [ ]:
com_keys = [k for k in maps if k.startswith('com_fit_')]
if com_keys:
    fig, axes = plt.subplots(1, len(com_keys), figsize=(5 * len(com_keys), 4.5),
                             squeeze=False)
    for ax, key in zip(axes[0], com_keys):
        a = np.asarray(M['starling'][key], float)
        b = np.asarray(M['darfix'][key], float)
        m = np.isfinite(a) & np.isfinite(b)
        ax.scatter(b[m], a[m], s=6, alpha=0.5)
        lo = float(np.nanmin([a[m].min(), b[m].min()]))
        hi = float(np.nanmax([a[m].max(), b[m].max()]))
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
        ax.set_xlabel('darfix'); ax.set_ylabel('starling')
        ax.set_title(f"{key}  r={fmt(maps[key].get('pearson_r'))}")
    plt.tight_layout(); plt.show()
else:
    print('no fit-vs-fit COM maps in this run')

## Reading the numbers

* **COM (fit-vs-fit)** should agree closely (high Pearson r, most/all pixels within one motor step) -- both packages fit a Gaussian centre, so any residual difference is dominated by the background estimate.
* **FWHM** is the map most sensitive to the modelling differences: the background estimator sets the baseline the peak sits on, and starling's constant-background Gaussian vs darfix's min-background multivariate Gaussian trade off width differently. A consistent-sign `mean_diff` here is the expected systematic offset, not a bug.
* **COM (moment-vs-moment)** carries darfix's median smoothing, so it is intentionally *not* identical to starling's unsmoothed moments; where the underlying COM barely varies, the Pearson r can be low even though the absolute differences are tiny (read `median_abs_diff` / `frac_within_1_step`, not r, in that regime).
* **`native_mosaicity`** (starling's covariance-based scalar mosaicity) is stored in `maps.h5` under `starling/native_mosaicity` for reference; it is not directly comparable to darfix, which is why `spread_fit` uses a common FWHM-based proxy instead.

The `notes` array in `report.json` restates these expected difference sources alongside the numbers.